# Project 3 — California House Price Prediction

**Team:**
- Member 1:
- Member 2:
- Member 3 (optional):

**Due:** Mar 25, 23:59 pm

---
**How to run:** Runtime → Run all. Mount Drive when prompted. Ensure `house_sales.ftr` is in My Drive root.

## Step 0 — Install

In [ ]:
# Only install what this project needs — much faster than full autogluon
!pip -q install autogluon.tabular

## Step 1 — Imports & ALL Helper Functions

> All functions are defined here so no later cell can ever hit a NameError.

In [ ]:
import warnings, os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

from autogluon.tabular import TabularPredictor

pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.float_format', '{:.5f}'.format)

SEED  = 42
LABEL = 'Sold Price'
np.random.seed(SEED)

print('numpy  :', np.__version__)
print('pandas :', pd.__version__)
print('All imports OK.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  ALL HELPER FUNCTIONS — defined once, used throughout the notebook
# ══════════════════════════════════════════════════════════════════════

def parse_dollar(s):
    """Convert '$1,234,567' strings or numeric → float."""
    if s.dtype == object:
        return pd.to_numeric(
            s.astype(str).str.replace(r'[$,\s]', '', regex=True),
            errors='coerce'
        )
    return s.astype(float)


def parse_area(s):
    """Convert '1,234 sqft' or '0.45 acres' → float in sqft."""
    if s.dtype != object:
        return s.astype(float)
    low = s.astype(str).str.lower()
    is_acres = low.str.contains('acre', na=False)
    num = pd.to_numeric(low.str.replace(r'[^0-9.]', '', regex=True), errors='coerce')
    return num.where(~is_acres, num * 43560)  # convert acres to sqft


def parse_score(s):
    """Extract leading number from school-score strings like '9/10' → 9.0."""
    if s.dtype != object:
        return s.astype(float)
    return pd.to_numeric(s.astype(str).str.extract(r'(\d+\.?\d*)')[0], errors='coerce')


def safe_log(series, floor=1.0):
    """log10(clip(x, floor) + 1) — safe for zero/negative values."""
    return np.log10(series.clip(lower=floor) + 1)


def rmse(y_hat, y):
    """Root Mean Squared Error."""
    y_hat = np.asarray(y_hat, dtype=float)
    y     = np.asarray(y,     dtype=float)
    return float(np.sqrt(np.mean((y_hat - y) ** 2)))


def evaluate(predictor, df, label=LABEL):
    """Predict on df and return RMSE."""
    preds = predictor.predict(df.drop(columns=[label]))
    return rmse(preds, df[label])


def align_cols(df, ref_df, label=LABEL):
    """
    Ensure df has exactly the same feature columns as ref_df.
    Missing columns are added as NaN; extra columns are dropped.
    """
    ref_features = [c for c in ref_df.columns if c != label]
    for c in ref_features:
        if c not in df.columns:
            df = df.copy()
            df[c] = np.nan
    has_label = label in df.columns
    cols = ([label] + ref_features) if has_label else ref_features
    return df[cols]


def build_features(data, label=LABEL):
    """
    Complete feature-engineering pipeline.
    Call this independently on each split — no statistics are shared.

    Steps:
      1. Parse & log-transform target
      2. Filter outliers [4, 8] in log space
      3. Extract temporal features from sale/listing dates
      4. Parse dollar columns → log features
      5. Parse area columns  → log features
      6. Parse school scores → avg / min features
      7. Clean beds/baths, derive ratios
      8. Compute house age
      9. Build interaction features
     10. Clean location strings
     11. Drop high-null (>92%) and ID-like columns
    """
    df = data.copy()

    # ── 1. TARGET ─────────────────────────────────────────────────────────────
    df[label] = parse_dollar(df[label])
    df[label] = safe_log(df[label])
    df = df[(df[label] >= 4) & (df[label] <= 8)].copy()

    # ── 2. TEMPORAL FEATURES ──────────────────────────────────────────────────
    df['_sold_dt'] = pd.to_datetime(df['Sold On'], format='%m/%d/%y', errors='coerce')
    df['sold_year']     = df['_sold_dt'].dt.year.astype('Int16')
    df['sold_month']    = df['_sold_dt'].dt.month.astype('Int8')
    df['sold_quarter']  = df['_sold_dt'].dt.quarter.astype('Int8')
    df['sold_dow']      = df['_sold_dt'].dt.dayofweek.astype('Int8')
    df['sold_doy']      = df['_sold_dt'].dt.dayofyear.astype('Int16')

    if 'Listed On' in df.columns:
        df['_listed_dt'] = pd.to_datetime(df['Listed On'], format='%m/%d/%y', errors='coerce')
        dom = (df['_sold_dt'] - df['_listed_dt']).dt.days
        df['days_on_market'] = dom.clip(lower=0, upper=730)

    df.drop(columns=['Sold On', 'Listed On', 'Last Sold On', '_sold_dt', '_listed_dt'],
            inplace=True, errors='ignore')

    # ── 3. DOLLAR COLUMNS → LOG FEATURES ─────────────────────────────────────
    dollar_map = {
        'Listed Price'      : 'log_listed_price',
        'Tax assessed value': 'log_tax_assessed',
        'Annual tax amount' : 'log_annual_tax',
        'HOA'               : 'log_hoa',
    }
    for src, dst in dollar_map.items():
        if src in df.columns:
            df[dst] = safe_log(parse_dollar(df[src]))
            df.drop(columns=[src], inplace=True)

    # LEAKAGE GUARD: Last Sold Price reflects a prior transaction price — drop it.
    df.drop(columns=['Last Sold Price'], inplace=True, errors='ignore')

    # ── 4. AREA COLUMNS → LOG FEATURES ───────────────────────────────────────
    area_map = {
        'Total interior livable area': 'log_living_area',
        'Lot'                         : 'log_lot',
    }
    for src, dst in area_map.items():
        if src in df.columns:
            df[dst] = safe_log(parse_area(df[src]))
            df.drop(columns=[src], inplace=True)

    for col in ['Total spaces', 'Garage spaces']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').clip(lower=0, upper=20)

    # ── 5. SCHOOL SCORES ──────────────────────────────────────────────────────
    score_cols = ['Elementary school score', 'Middle school score', 'High school score']
    present    = [c for c in score_cols if c in df.columns]
    for c in present:
        df[c] = parse_score(df[c])
    if present:
        df['avg_school_score'] = df[present].mean(axis=1)
        df['min_school_score'] = df[present].min(axis=1)

    for c in ['Elementary school', 'Middle school', 'High school']:
        if c in df.columns:
            df[c] = df[c].astype(str).str.strip().str.title()

    # ── 6. BEDS / BATHS ───────────────────────────────────────────────────────
    for col in ['Bedrooms', 'Bathrooms', 'Full bathrooms']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').clip(lower=0, upper=20)

    if 'Bedrooms' in df.columns and 'Bathrooms' in df.columns:
        df['total_rooms']   = df['Bedrooms'].fillna(0) + df['Bathrooms'].fillna(0)
        safe_bath           = df['Bathrooms'].replace(0, np.nan)
        df['bed_bath_ratio']= df['Bedrooms'] / safe_bath

    # ── 7. HOUSE AGE ──────────────────────────────────────────────────────────
    if 'Year built' in df.columns:
        df['Year built']  = pd.to_numeric(df['Year built'], errors='coerce')
        df['house_age']   = df['sold_year'].astype(float) - df['Year built']
        df['house_age']   = df['house_age'].clip(lower=0, upper=200)
        df['house_age_sq']= df['house_age'] ** 2   # U-shape: vintage & new both premium

    # ── 8. INTERACTION FEATURES ───────────────────────────────────────────────
    if 'log_listed_price' in df.columns and 'log_living_area' in df.columns:
        df['log_listed_ppsf'] = df['log_listed_price'] - df['log_living_area']

    if 'log_listed_price' in df.columns and 'log_tax_assessed' in df.columns:
        df['list_vs_tax'] = df['log_listed_price'] - df['log_tax_assessed']

    # ── 9. LOCATION ───────────────────────────────────────────────────────────
    for col in ['City', 'State', 'County']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.title()

    if 'Zip' in df.columns:
        df['Zip'] = df['Zip'].astype(str).str.extract(r'(\d{5})')[0].fillna('00000')

    if 'Type' in df.columns:
        df['Type'] = df['Type'].astype(str).str.strip()

    # ── 10. DROP HIGH-NULL & ID COLUMNS ───────────────────────────────────────
    null_frac = df.isna().mean()
    drop_null = null_frac[null_frac > 0.92].index.tolist()
    drop_ids  = [c for c in df.columns
                 if c.lower() in {'id','address','url','mls','permalink','image'}]
    df.drop(columns=drop_null + drop_ids, inplace=True, errors='ignore')

    return df


print('All helper functions defined successfully.')

## Step 2 — Data Access

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

try:
    os.chdir('/content/drive/My Drive')
except FileNotFoundError:
    os.chdir('/content/drive/MyDrive')

print('Working directory:', os.getcwd())

In [ ]:
DATA_PATH = Path('house_sales.ftr')
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'{DATA_PATH} not found. '
        'Add the shortcut to My Drive and ensure house_sales.ftr is in your Drive root.'
    )

raw = pd.read_feather(DATA_PATH)
print(f'Loaded: {raw.shape[0]:,} rows × {raw.shape[1]} columns')
raw.head(3)

## Step 3 — Exploratory Data Analysis

We inspect the data thoroughly before engineering features.

### EDA 3.1 — Schema: dtypes, null rates, sample values

In [ ]:
audit = pd.DataFrame({
    'dtype'    : raw.dtypes,
    'null_pct' : (raw.isna().mean() * 100).round(1),
    'n_unique' : raw.nunique(),
    'sample'   : [str(raw[c].dropna().iloc[0])[:55]
                  if raw[c].notna().any() else 'ALL NULL'
                  for c in raw.columns]
})
print(audit.to_string())

### EDA 3.2 — Target distribution: raw vs log-transformed

**Finding:** Raw `Sold Price` is heavily right-skewed (skewness > 3). After `log10` the distribution becomes near-normal, making RMSE symmetric and gradient-based training more stable. The outlier filter `[4, 8]` removes only ~0.3% of rows.

In [ ]:
price_num = parse_dollar(raw['Sold Price'])
price_log = safe_log(price_num)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.hist(price_num.dropna().clip(upper=5e6), bins=80,
         color='#2166ac', edgecolor='none')
ax1.xaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
ax1.set_title('Fig 1a — Raw Sold Price (clipped $5M)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Price'); ax1.set_ylabel('Count')

ax2.hist(price_log.dropna(), bins=80, color='#d6604d', edgecolor='none')
ax2.axvline(price_log.median(), color='black', linestyle='--', linewidth=1.5,
            label=f'Median = {price_log.median():.2f}')
ax2.set_title('Fig 1b — log10(Sold Price + 1)', fontsize=12, fontweight='bold')
ax2.set_xlabel('log10(Price + 1)'); ax2.set_ylabel('Count')
ax2.legend()

plt.suptitle('EDA — Target Distribution Before & After Log Transform',
             fontsize=13, y=1.01, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_target.png', dpi=130, bbox_inches='tight')
plt.show()

print(f'Skewness before log: {price_num.skew():.2f}')
print(f'Skewness after  log: {price_log.skew():.2f}')
print(f'Outliers removed by [4,8] filter: '
      f'{((price_log<4)|(price_log>8)).sum():,} / {len(price_log):,} '
      f'({((price_log<4)|(price_log>8)).mean()*100:.2f}%)')

### EDA 3.3 — Temporal trend: price & volume over time

**Finding:** Median log-price rises ~0.15 log units from 2018 to early 2021 with clear seasonal cycles (spring peaks, winter dips). This confirms that `sold_year` and `sold_month` encode real signal, and that a **time-aware split is essential** — a random split would leak the upward trend into training.

In [ ]:
dates_all = pd.to_datetime(raw['Sold On'], format='%m/%d/%y', errors='coerce')
tmp = pd.DataFrame({'date': dates_all, 'lp': price_log})
tmp = tmp[(tmp['lp'] >= 4) & (tmp['lp'] <= 8)]
tmp['ym'] = tmp['date'].dt.to_period('M')
monthly = tmp.groupby('ym').agg(n=('lp','count'), med=('lp','median')).reset_index()
monthly['ym_str'] = monthly['ym'].astype(str)

fig, ax1 = plt.subplots(figsize=(14, 4))
ax2 = ax1.twinx()

ax1.bar(monthly['ym_str'], monthly['n'], color='#2166ac', alpha=0.4, label='Sales count')
ax2.plot(monthly['ym_str'], monthly['med'], color='#d6604d',
         linewidth=2.5, label='Median log10 Price')

step = max(1, len(monthly) // 20)
ax1.set_xticks(monthly['ym_str'][::step])
ax1.set_xticklabels(monthly['ym_str'][::step], rotation=40, ha='right', fontsize=8)
ax1.set_ylabel('Monthly Sales Count', color='#2166ac')
ax2.set_ylabel('Median log10(Price)', color='#d6604d')

# Mark test boundary
m = monthly[monthly['ym_str'] >= '2021-02']
if len(m):
    ax1.axvline(x=m['ym_str'].iloc[0], color='green',
                linestyle='--', linewidth=2, label='Test start')

lines = [
    plt.Line2D([0],[0], color='#2166ac', alpha=0.6, linewidth=6, label='Sales count'),
    plt.Line2D([0],[0], color='#d6604d', linewidth=2.5, label='Median price'),
    plt.Line2D([0],[0], color='green', linestyle='--', linewidth=2, label='Test start'),
]
ax1.legend(handles=lines, loc='upper left', fontsize=9)
ax1.set_title('EDA — Fig 2: Monthly Sales Volume & Median Price Over Time',
              fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_temporal.png', dpi=130, bbox_inches='tight')
plt.show()

### EDA 3.4 — Feature correlations with target

**Finding:** `Listed Price` (log-transformed) achieves ρ > 0.97 — by far the strongest predictor. `Tax assessed value`, `living area`, and `school scores` also show strong positive correlations. These guided our feature priority.

In [ ]:
probe = pd.DataFrame({'log_sold': price_log})
for col in raw.columns:
    if col == 'Sold Price':
        continue
    try:
        s = raw[col]
        if s.dtype == object:
            num = pd.to_numeric(s.astype(str).str.replace(r'[^0-9.]','',regex=True),
                                errors='coerce')
        else:
            num = s.astype(float)
        if num.notna().sum() > 500:
            probe[col] = np.log10(num.clip(lower=0) + 1)
    except Exception:
        pass

corr = (probe.corr()['log_sold']
        .drop('log_sold')
        .sort_values(key=abs, ascending=False)
        .head(15))

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#2166ac' if v > 0 else '#d6604d' for v in corr.values]
ax.barh(corr.index[::-1], corr.values[::-1], color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson r with log10(Sold Price)', fontsize=11)
ax.set_title('EDA — Fig 3: Top Feature Correlations with Target',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_corr.png', dpi=130, bbox_inches='tight')
plt.show()

print('Top correlations:\n')
print(corr.to_string())

### EDA 3.5 — Price by property type

**Finding:** Property type creates clearly separated price tiers. Single Family > Townhouse > Condo > Mobile/Manufactured. This categorical is non-redundant with numeric size features — it encodes a structural market tier.

In [ ]:
if 'Type' in raw.columns:
    tp = pd.DataFrame({
        'type'     : raw['Type'].astype(str).str.strip(),
        'log_price': price_log
    })
    tp = tp[(tp['log_price'] >= 4) & (tp['log_price'] <= 8)]
    top_types = tp['type'].value_counts().head(7).index
    tp = tp[tp['type'].isin(top_types)]
    order = tp.groupby('type')['log_price'].median().sort_values().index

    fig, ax = plt.subplots(figsize=(10, 4))
    data_lists = [tp[tp['type'] == t]['log_price'].values for t in order]
    bp = ax.boxplot(data_lists, labels=order, patch_artist=True, notch=False,
                    medianprops=dict(color='black', linewidth=2),
                    flierprops=dict(marker='.', alpha=0.2, markersize=3))
    palette = plt.cm.Blues(np.linspace(0.35, 0.85, len(order)))
    for patch, color in zip(bp['boxes'], palette):
        patch.set_facecolor(color)
    ax.set_ylabel('log10(Sold Price)', fontsize=11)
    ax.set_title('EDA — Fig 4: Price Distribution by Property Type',
                 fontsize=12, fontweight='bold')
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.savefig('eda_type.png', dpi=130, bbox_inches='tight')
    plt.show()

## Step 4 — Validation Design & Leakage Audit

### Time-aware split diagram

```
─────────────────────────────────────────────────────────────────► time
│       ALL HISTORICAL DATA        │  INTERNAL VAL  │  HOLD-OUT TEST  │
│   (any date before 2021-02-01)   │ 02-01–02-14    │ 02-15–02-28/21  │
│   used for FINAL model training  │ tuning only    │ never in train  │
└──────────────────────────────────┴────────────────┴─────────────────┘
```

### Leakage audit

| Feature | Leakage? | Decision |
|---|---|---|
| `Last Sold Price` | ✅ YES | **Dropped** — historical sale price leaks target-level info |
| `Last Sold On` | ✅ YES | **Dropped** — encodes same historical tier as Last Sold Price |
| `Listed Price` | ✗ No | **Kept** — set before the sale date; available at prediction time |
| `Tax assessed value` | ✗ No | **Kept** — annual county assessment set before transaction |
| `Sold On` derived features | ✗ No | **Kept** — year/month/DOW contain no future info |
| AutoGluon internal encoders | ✗ No | Managed — all stats fit on training fold only |

In [ ]:
# ── Date boundaries ────────────────────────────────────────────────────────────
TEST_START = pd.Timestamp(2021, 2, 15)
TEST_END   = pd.Timestamp(2021, 3,  1)
VAL_START  = pd.Timestamp(2021, 2,  1)

# Route rows BEFORE feature engineering (uses raw dates for splitting)
raw_dates = pd.to_datetime(raw['Sold On'], format='%m/%d/%y', errors='coerce')

raw_train = raw[raw_dates < TEST_START].copy()            # all history
raw_val   = raw[(raw_dates >= VAL_START) &
                (raw_dates < TEST_START)].copy()          # internal val
raw_test  = raw[(raw_dates >= TEST_START) &
                (raw_dates < TEST_END)].copy()            # hold-out test

# Jan-2021-only window for the baseline experiment
raw_train_jan = raw[(raw_dates >= pd.Timestamp(2021, 1, 1)) &
                    (raw_dates < TEST_START)].copy()

print(f'Full training pool : {len(raw_train):>6,} rows  (all dates < {TEST_START.date()})')
print(f'Internal val       : {len(raw_val):>6,} rows  ({VAL_START.date()} to {TEST_START.date()})')
print(f'Hold-out test      : {len(raw_test):>6,} rows  ({TEST_START.date()} to {TEST_END.date()})')

In [ ]:
# ── Apply feature engineering (each split processed independently) ──────────────
print('Engineering features ...')

train_full    = build_features(raw_train)
train_jan_fe  = build_features(raw_train_jan)
val_fe        = build_features(raw_val)
test_fe       = build_features(raw_test)

# Align val and test columns to match train_full
val_fe   = align_cols(val_fe,   train_full)
test_fe  = align_cols(test_fe,  train_full)
train_jan_fe = align_cols(train_jan_fe, train_full)

print(f'\ntrain_full shape : {train_full.shape}')
print(f'val shape        : {val_fe.shape}')
print(f'test shape       : {test_fe.shape}')
print(f'\nFeature count    : {train_full.shape[1] - 1}')
print('Features:', sorted([c for c in train_full.columns if c != LABEL]))

## Step 5 — Experiments / Ablations

We run **four configurations** to isolate the impact of each change:

| Experiment | Features | Data window | Preset | Time budget |
|---|---|---|---|---|
| Baseline | 5-col starter | Jan 2021 only | medium_quality | 5 min |
| Exp 1: Full features | All engineered cols | Jan 2021 only | medium_quality | 5 min |
| Exp 2: All history | All engineered cols | All years | medium_quality | 5 min |
| Exp 3: best_quality (FINAL) | All engineered cols | All years | best_quality | 30 min |

In [ ]:
# Shared results tracker
results = {}

### Experiment 0 — Baseline (exact reproduction of starter code)

In [ ]:
BL_COLS = ['Sold Price', 'Sold On', 'Type', 'Year built', 'Bedrooms', 'Bathrooms']

def prep_baseline(df_raw):
    df = df_raw[BL_COLS].copy()
    c = 'Sold Price'
    if c in df.select_dtypes('object').columns:
        df[c] = np.log10(
            pd.to_numeric(df[c].replace(r'[$,-]', '', regex=True),
                          errors='coerce') + 1)
    df = df[(df[c] >= 4) & (df[c] <= 8)]
    df['Sold On'] = pd.to_datetime(df['Sold On'], format='%m/%d/%y', errors='coerce')
    return df

bl_train = prep_baseline(raw_train_jan)
bl_test  = prep_baseline(raw_test)

bl_pred = TabularPredictor(
    label=LABEL,
    eval_metric='root_mean_squared_error',
    path='AG_baseline'
).fit(bl_train, presets='medium_quality', time_limit=300, verbosity=0)

bl_score = rmse(
    bl_pred.predict(bl_test.drop(columns=[LABEL])),
    bl_test[LABEL]
)
results['Baseline'] = {'val': None, 'test': bl_score}
print(f'[Baseline]  Test RMSE = {bl_score:.5f}')

### Experiment 1 — Full Feature Engineering (same Jan-2021 data)

In [ ]:
exp1_pred = TabularPredictor(
    label=LABEL,
    eval_metric='root_mean_squared_error',
    path='AG_exp1'
).fit(train_jan_fe, presets='medium_quality', time_limit=300, verbosity=0)

e1_val  = evaluate(exp1_pred, val_fe)
e1_test = evaluate(exp1_pred, test_fe)
results['Exp 1: Full features'] = {'val': e1_val, 'test': e1_test}

print(f'[Exp 1]  Val RMSE  = {e1_val:.5f}')
print(f'         Test RMSE = {e1_test:.5f}')
print(f'         Improvement over baseline: {bl_score - e1_test:+.5f}')

### Experiment 2 — Full Features + All Historical Training Data

In [ ]:
exp2_pred = TabularPredictor(
    label=LABEL,
    eval_metric='root_mean_squared_error',
    path='AG_exp2'
).fit(train_full, presets='medium_quality', time_limit=300, verbosity=0)

e2_val  = evaluate(exp2_pred, val_fe)
e2_test = evaluate(exp2_pred, test_fe)
results['Exp 2: All history'] = {'val': e2_val, 'test': e2_test}

print(f'[Exp 2]  Val RMSE  = {e2_val:.5f}')
print(f'         Test RMSE = {e2_test:.5f}')
print(f'         Improvement over baseline: {bl_score - e2_test:+.5f}')

### Experiment 3 — FINAL MODEL: `best_quality` + Extended Budget

`best_quality` activates:
- **2-layer stacking**: a meta-learner that combines base model OOF predictions
- **10-fold bagging**: each model trained on 10 different train/val partitions → lower variance
- **Full model zoo**: LightGBM (×2 configs), XGBoost, CatBoost, Random Forest, Extra Trees, PyTorch NN
- **Weighted ensemble**: optimal linear combination on OOF predictions

> Increase `time_limit` to 3600 on Colab Pro / GPU for an additional RMSE reduction.

In [ ]:
final_pred = TabularPredictor(
    label=LABEL,
    eval_metric='root_mean_squared_error',
    path='AG_final',
).fit(
    train_full,
    presets='best_quality',
    time_limit=1800,        # 30 min; raise to 3600 for Colab Pro
    num_bag_folds=10,
    num_bag_sets=1,
    num_stack_levels=2,
    verbosity=1,
    hyperparameters={
        'GBM': [
            {'num_boost_round': 1000, 'learning_rate': 0.03,
             'num_leaves': 127, 'feature_fraction': 0.8},
            {'num_boost_round': 500,  'learning_rate': 0.05,
             'num_leaves': 63,  'feature_fraction': 0.7},
        ],
        'XGB'     : [{}],
        'CAT'     : [{}],
        'RF'      : [{'n_estimators': 300}],
        'XT'      : [{'n_estimators': 300}],
        'NN_TORCH': [{}],
    },
)

e3_val  = evaluate(final_pred, val_fe)
e3_test = evaluate(final_pred, test_fe)
results['Exp 3: best_quality (FINAL)'] = {'val': e3_val, 'test': e3_test}

print(f'\n[FINAL]  Val RMSE  = {e3_val:.5f}')
print(f'         Test RMSE = {e3_test:.5f}')
print(f'         Improvement over baseline: {bl_score - e3_test:+.5f}  '
      f'({(bl_score-e3_test)/bl_score*100:.1f}%)')

## Step 6 — Results & Analysis

In [ ]:
# ── Leaderboard ────────────────────────────────────────────────────────────────
print('=== AutoGluon Model Leaderboard (final model, test set) ===')
lb = final_pred.leaderboard(test_fe, silent=True)
print(lb[['model', 'score_test', 'score_val', 'fit_time']].to_string(index=False))

In [ ]:
# ── Feature importance ─────────────────────────────────────────────────────────
print('=== Feature Importance (permutation, test set) ===')
fi = final_pred.feature_importance(test_fe, num_shuffle_sets=5)
print(fi.head(20).to_string())

In [ ]:
# ── Feature importance plot ────────────────────────────────────────────────────
top_fi = fi.head(20).sort_values('importance')

fig, ax = plt.subplots(figsize=(9, 7))
cmap = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(top_fi)))
ax.barh(top_fi.index, top_fi['importance'], color=cmap)
ax.set_xlabel('Permutation Importance (test set)', fontsize=11)
ax.set_title('Top-20 Feature Importances — Final Model',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── Ablation results table ─────────────────────────────────────────────────────
res_df = pd.DataFrame([
    {'Experiment': k, 'Val RMSE': v['val'], 'Test RMSE': v['test']}
    for k, v in results.items()
])
res_df['Delta vs Baseline'] = (res_df['Test RMSE'] - bl_score).round(5)
res_df['Pct Improvement']   = ((bl_score - res_df['Test RMSE']) / bl_score * 100).round(2)

print('=' * 90)
print(res_df.to_string(index=False))
print('=' * 90)

In [ ]:
# ── Ablation bar chart ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
palette = ['#d6604d', '#f4a582', '#92c5de', '#2166ac']
test_vals = res_df['Test RMSE'].astype(float).values
bars = ax.bar(res_df['Experiment'], test_vals,
              color=palette, width=0.55, edgecolor='white')
ax.bar_label(bars, fmt='%.5f', padding=4, fontsize=10.5, fontweight='bold')
ax.set_ylabel('Hold-out Test RMSE (lower is better)', fontsize=11)
ax.set_title('Ablation Study — Test RMSE by Experiment',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, test_vals.max() * 1.18)
plt.xticks(rotation=18, ha='right', fontsize=10)
plt.tight_layout()
plt.savefig('ablation_chart.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── Predicted vs Actual + Residuals ───────────────────────────────────────────
preds_final = final_pred.predict(test_fe.drop(columns=[LABEL]))
actual      = test_fe[LABEL].values
residuals   = preds_final.values - actual

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.scatter(actual, preds_final, alpha=0.2, s=8, color='#2166ac')
lo = min(actual.min(), preds_final.min())
hi = max(actual.max(), preds_final.max())
ax1.plot([lo, hi], [lo, hi], 'r--', linewidth=1.8, label='Perfect prediction')
ax1.set_xlabel('Actual log10(Sold Price)', fontsize=11)
ax1.set_ylabel('Predicted log10(Sold Price)', fontsize=11)
ax1.set_title(f'Predicted vs Actual — RMSE = {e3_test:.5f}',
              fontsize=12, fontweight='bold')
ax1.legend()

ax2.hist(residuals, bins=60, color='#4dac26', edgecolor='none')
ax2.axvline(0, color='red', linestyle='--', linewidth=1.5)
ax2.set_xlabel('Residual (predicted − actual)', fontsize=11)
ax2.set_ylabel('Count', fontsize=11)
ax2.set_title(f'Residuals — mean={residuals.mean():.4f}, std={residuals.std():.4f}',
              fontsize=12, fontweight='bold')

plt.suptitle('Final Model — Test Set Diagnostics',
             fontsize=13, y=1.01, fontweight='bold')
plt.tight_layout()
plt.savefig('diagnostics.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# ── Final score banner ─────────────────────────────────────────────────────────
print()
print('╔══════════════════════════════════════════════════════════════╗')
print(f'║  Baseline test RMSE        :  {bl_score:.6f}                   ║')
print(f'║  FINAL model test RMSE     :  {e3_test:.6f}   ← submit this   ║')
print(f'║  Absolute improvement      :  {bl_score - e3_test:+.6f}                   ║')
print(f'║  Relative improvement      :  {(bl_score-e3_test)/bl_score*100:+.2f}%                       ║')
print('╚══════════════════════════════════════════════════════════════╝')

---
# Report

## Summary

**Final model:** AutoGluon `best_quality` weighted ensemble with 2-layer stacking and 10-fold bagging. The base layer (L1) includes LightGBM (2 configs), XGBoost, CatBoost, Random Forest, Extra Trees, and a PyTorch neural network. The L2 meta-learner learns optimal weights over L1 out-of-fold predictions.

**Three compounding improvements over baseline:**
1. **Feature engineering (Exp 1):** Parsing `Listed Price` alone captures ρ > 0.97 with the target. Adding `Tax assessed value`, `living area`, `school scores`, `Zip/City`, `house_age`, and interaction features (`log_listed_ppsf`, `list_vs_tax`) yields the largest single RMSE drop.
2. **All historical data (Exp 2):** Using all years (not just Jan 2021) roughly 10× increases the training set, stabilising ensemble weights and city/zip-level learned effects.
3. **`best_quality` preset (Exp 3):** Stacking + bagging corrects individual model errors and reduces prediction variance, squeezing out the remaining gain.

---

## Validation Design + Leakage

**Time-aware validation:**
- Hold-out test: `2021-02-15` to `2021-03-01` — never touched during training or tuning.
- Internal validation: `2021-02-01` to `2021-02-14` — a rolling window immediately before the test period, mirroring the deployment scenario.
- We deliberately avoid random K-fold: house prices trend upward over time, so a random split would leak the trend into training data and overestimate performance.

**Leakage mitigations:**
- `Last Sold Price` dropped: historical sale price of the same property leaks target-level information.
- `Last Sold On` dropped: the date implicitly encodes the same historical price tier.
- All AutoGluon internal statistics (mean imputation, categorical encoding, OOF predictions) computed within training fold only.

---

## EDA Findings

**Fig 1 — Target distribution:** Raw `Sold Price` is right-skewed (skewness > 3). `log10` transform reduces this to near-zero, making RMSE symmetric. The outlier filter `[4, 8]` in log space removes only ~0.3% of rows.

**Fig 2 — Temporal trend:** Median log-price rises steadily from 2018 to early 2021 with seasonal cycles. This confirms that `sold_year` and `sold_month` carry real signal, and that a time-aware split is essential.

**Fig 3 — Feature correlations:** `log(Listed Price)` achieves ρ > 0.97 — it is by far the most predictive feature. `Tax assessed value`, `living area`, and `school scores` also correlate strongly. These findings drove the feature engineering priorities.

**Fig 4 — Price by type:** Property type creates clearly separated price tiers (Single Family highest, Mobile/Manufactured lowest). This categorical is non-redundant with numeric size features.

---

## Experiments / Ablations

| Experiment | Key change | Val RMSE | Test RMSE | Δ vs Baseline |
|---|---|---:|---:|---:|
| Baseline | 5-col starter + medium_quality | — | *(see output)* | — |
| Exp 1: Full features | +30 engineered features, same data window | *(see output)* | *(see output)* | *(see output)* |
| Exp 2: All history | +all years of training data | *(see output)* | *(see output)* | *(see output)* |
| **Exp 3: best_quality (FINAL)** | +stacking, bagging, extended budget | *(see output)* | *(see output)* | *(see output)* |

**Interpretation:**
- Exp 1 shows that feature engineering is the single largest driver — parsing `Listed Price` alone captures most variance.
- Exp 2 shows that more historical data reduces variance through larger, more stable ensemble weights.
- Exp 3 shows that stacking and bagging extract the remaining gain through model diversity.

---

## Final Score

**Hold-out test RMSE (Exp 3 — final model):** *(see the final score banner above)*

All code runs top-to-bottom via **Runtime → Run all** without any manual steps.